In [2]:
import os
import assemblyai as aai
import json
from pathlib import Path
from datetime import timedelta

aai.settings.api_key = "8776ba356c4241d9b2631001ef3dcc4f"  

def format_timestamp(milliseconds):
    """Convert milliseconds to HH:MM:SS format"""
    seconds = milliseconds / 1000
    td = timedelta(seconds=seconds)
    hours, remainder = divmod(td.total_seconds(), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"

def extract_transcript_assemblyai(video_path):
    """Extract transcript using AssemblyAI SDK"""
    print(f"Processing: {video_path}")
    
    # simple transcription 
    transcriber = aai.Transcriber()
    transcript = transcriber.transcribe(str(video_path))
    
    # check for errors
    if transcript.status == aai.TranscriptStatus.error:
        raise RuntimeError(f"Transcription failed: {transcript.error}")
    
    transcript_data = {
        "file": str(video_path),
        "confidence": getattr(transcript, 'confidence', 0),
        "language": getattr(transcript, 'language_code', "unknown"),
        "text": getattr(transcript, 'text', ''),
        "segments": []
    }
    
    if hasattr(transcript, 'words') and transcript.words:
        current_sentence = []
        current_start = None
        
        for word in transcript.words:
            if current_start is None:
                current_start = word.start
            
            current_sentence.append(word.text)
            
            if (word.text.endswith(('.', '!', '?', ':')) or 
                len(current_sentence) >= 15):
                
                segment_data = {
                    "start": current_start / 1000,  # convert to seconds
                    "end": word.end / 1000,
                    "start_formatted": format_timestamp(current_start),
                    "end_formatted": format_timestamp(word.end),
                    "text": " ".join(current_sentence).strip(),
                    "confidence": getattr(word, 'confidence', 0)
                }
                transcript_data["segments"].append(segment_data)
                
                current_sentence = []
                current_start = None
        
        # handle remaining words
        if current_sentence and current_start is not None:
            last_word = transcript.words[-1]
            segment_data = {
                "start": current_start / 1000,
                "end": last_word.end / 1000,
                "start_formatted": format_timestamp(current_start),
                "end_formatted": format_timestamp(last_word.end),
                "text": " ".join(current_sentence).strip(),
                "confidence": getattr(last_word, 'confidence', 0)
            }
            transcript_data["segments"].append(segment_data)
    
    if not transcript_data["segments"] and transcript_data["text"]:
        import re
        sentences = re.split(r'[.!?]+', transcript_data["text"])
        
        for i, sentence in enumerate(sentences):
            sentence = sentence.strip()
            if sentence:
                segment_data = {
                    "start": i * 5,  # Rough estimate: 5 seconds per sentence
                    "end": (i + 1) * 5,
                    "start_formatted": format_timestamp(i * 5000),
                    "end_formatted": format_timestamp((i + 1) * 5000),
                    "text": sentence,
                    "confidence": transcript_data["confidence"]
                }
                transcript_data["segments"].append(segment_data)
    
    return transcript_data

def process_video_folder_assemblyai(folder_path, output_folder="transcripts"):
    """Process all videos using AssemblyAI SDK"""
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v', '.mp3', '.wav', '.m4a'}
    
    folder_path = Path(folder_path)
    output_path = Path(output_folder)
    output_path.mkdir(exist_ok=True)
    
    media_files = []
    for ext in video_extensions:
        media_files.extend(folder_path.glob(f"*{ext}"))
        media_files.extend(folder_path.glob(f"*{ext.upper()}"))
    
    if not media_files:
        print(f"No media files found in {folder_path}")
        return
    
    print(f"Found {len(media_files)} media files")
    
    for media_file in media_files:
        try:
            print(f"\n{'='*50}")
            print(f"Processing: {media_file.name}")
            print(f"{'='*50}")
            
            # extract transcript
            transcript = extract_transcript_assemblyai(media_file)
            
            # save as JSON
            output_file = output_path / f"{media_file.stem}_transcript.json"
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(transcript, f, indent=2, ensure_ascii=False)
            
            # save as readable text format
            txt_file = output_path / f"{media_file.stem}_transcript.txt"
            with open(txt_file, 'w', encoding='utf-8') as f:
                f.write(f"Transcript for: {media_file.name}\n")
                f.write(f"Confidence: {transcript['confidence']:.2f}\n")
                f.write(f"Language: {transcript['language']}\n")
                f.write("=" * 50 + "\n\n")
                
                for segment in transcript["segments"]:
                    f.write(f"[{segment['start_formatted']} - {segment['end_formatted']}]\n")
                    f.write(f"{segment['text']}\n\n")
            
            print(f"Completed: {media_file.name}")
            print(f"  - Confidence: {transcript['confidence']:.2f}")
            print(f"  - JSON: {output_file}")
            print(f"  - TXT: {txt_file}")
            
        except Exception as e:
            print(f"✗ Error processing {media_file.name}: {str(e)}")

def main():
    videos_folder = "/Users/sooyeonkim/Desktop/MIDS/DATA266/final_project/videos"
    output_folder = "transcripts"
    
    if not os.path.exists(videos_folder):
        print(f"Videos folder '{videos_folder}' not found!")
        return
    
    print("AssemblyAI Video Transcript Extractor (Updated)")
    print("=" * 40)
    print(f"Input folder: {videos_folder}")
    print(f"Output folder: {output_folder}")
    print()
    
    process_video_folder_assemblyai(videos_folder, output_folder)
    print("\nAll processing complete!")

if __name__ == "__main__":
    main()

AssemblyAI Video Transcript Extractor (Updated)
Input folder: /Users/sooyeonkim/Desktop/MIDS/DATA266/final_project/videos
Output folder: transcripts

Found 4 media files

Processing: Cholecystectomy by Cal Shipley, M.D..mp4
Processing: /Users/sooyeonkim/Desktop/MIDS/DATA266/final_project/videos/Cholecystectomy by Cal Shipley, M.D..mp4
Completed: Cholecystectomy by Cal Shipley, M.D..mp4
  - Confidence: 0.98
  - JSON: transcripts/Cholecystectomy by Cal Shipley, M.D._transcript.json
  - TXT: transcripts/Cholecystectomy by Cal Shipley, M.D._transcript.txt

Processing: Laparoscopic Cholecystectomy Full HD Video.mp4
Processing: /Users/sooyeonkim/Desktop/MIDS/DATA266/final_project/videos/Laparoscopic Cholecystectomy Full HD Video.mp4
✗ Error processing Laparoscopic Cholecystectomy Full HD Video.mp4: Server disconnected without sending a response.

Processing: Laparoscopic cholecystectomy for Mirizzi syndrome.mp4
Processing: /Users/sooyeonkim/Desktop/MIDS/DATA266/final_project/videos/Laparosco